In [ ]:
# ===================================================================================
# WEB MAP MIGRATION V34.3 (STRICT SHARING + GROUPS + THUMBNAILS)
# - FIX: Corrected Folder object access logic and added Root fallback.
# - FIX: Solves "SharingGroupManager has no len()" crash.
# - UPDATE: Strict Sharing Mirroring (Private stays Private).
# - UPDATE: Mirrors Group Sharing (Matches Target Groups by Title).
# ===================================================================================

import pandas as pd
import json
import copy
import time
import csv
import os
import datetime
import urllib3
import requests
import sys
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- NOTEBOOK-SPECIFIC ---
# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_2_webmaps.json")
if os.path.exists(_sidecar_path):
    import json as _json
    WEB_MAP_IDS_TO_MIGRATE = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(WEB_MAP_IDS_TO_MIGRATE)} Web Map IDs from sidecar.")
else:
    WEB_MAP_IDS_TO_MIGRATE = [
        # Example Source ID's
        # "5f879c9d764e477286a171ad541cb65b", 
        # "8eb42218bad84b6791c24b0a0654928a", 
    ]

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
source_gis = None
target_gis = None

print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount('https://', adapter)
    
    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)
    
    print(f"   > Source Connected: {source_gis.url}")
    if not target_gis.users.me: raise Exception("Target login failed.")
    print(f"   > Target Connected: {target_gis.users.me.username}")

except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    sys.exit(1)

if not source_gis or not target_gis:
    print("❌ GIS Initialization incomplete. Aborting.")
    sys.exit(1)

# =============================================================================
# --- LOAD LEDGER --------------------------------------------------------------
# =============================================================================
ID_MAP = {}
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns and "TargetID" in df_log.columns:
            for _, row in df_log.iterrows():
                s_id = str(row["SourceID"]).strip()
                t_id = str(row["TargetID"]).strip()
                if s_id and t_id and s_id != "nan" and t_id != "nan":
                    ID_MAP[s_id] = t_id
        print(f"✅ Loaded Ledger: {len(ID_MAP)} ID mappings found.")
    except Exception as e:
        print(f"⚠️ Failed to load CSV: {e}")

STATS = {"maps_scanned": 0, "maps_created": 0, "failures": 0, "maps_skipped_log": 0, "maps_skipped_missing_data": 0, "layers_rewired": 0}
ALREADY_MIGRATED_IDS = set()

if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
    except: pass

def log_migration(source_id, index, target_id, title, item_type):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, index, target_id, title,
                datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), item_type
            ])
            ALREADY_MIGRATED_IDS.add(str(source_id))
    except: pass

# =============================================================================
# --- FIX: ENSURE ADMIN FOLDER EXISTS (ROBUST VERSION) -------------------------
# =============================================================================
ACTIVE_FOLDER = DEFAULT_FOLDER

def verify_admin_default_folder():
    """
    Checks if the default staging folder exists. 
    Handles both dict and object return types from me.folders.
    If it fails, it disables the folder (Sets to None) so the script won't crash.
    """
    global ACTIVE_FOLDER
    try:
        me = target_gis.users.me
        folder_names = []
        
        # Handle both object and dict types safely
        for f in me.folders:
            if isinstance(f, dict):
                folder_names.append(f.get('title'))
            elif hasattr(f, 'title'):
                folder_names.append(f.title)
        
        if DEFAULT_FOLDER not in folder_names:
            print(f"   [Setup] Creating missing admin folder: '{DEFAULT_FOLDER}'...")
            target_gis.content.create_folder(DEFAULT_FOLDER)
        else:
            print(f"   [Setup] Admin folder '{DEFAULT_FOLDER}' verified.")
            
    except Exception as e:
        print(f"   ⚠️ Warning: Could not verify/create admin folder. Falling back to ROOT. Error: {e}")
        ACTIVE_FOLDER = None # FALLBACK TO ROOT TO PREVENT CRASH

# Call this immediately after connection
verify_admin_default_folder()

# =============================================================================
# --- HELPERS (METADATA, OWNER, FOLDERS) ---------------------------------------
# =============================================================================
def copy_full_metadata(source_item, target_item, source_id):
    try:
        final_tags = list(source_item.tags or [])
        if "Migrated" not in final_tags: final_tags.append("Migrated")
        tag_marker = f"SourceID:{source_id}"
        if tag_marker not in final_tags: final_tags.append(tag_marker)
        
        update_props = {
            "title": target_item.title,
            "tags": final_tags,
            "snippet": source_item.snippet,
            "description": source_item.description,
            "accessInformation": source_item.accessInformation,
            "licenseInfo": source_item.licenseInfo
        }
        if getattr(source_item, "categories", None):
            update_props["categories"] = source_item.categories
        target_item.update(item_properties=update_props)
    except: pass

def get_source_folder_name(source_item):
    try:
        if not source_item.ownerFolder: return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                # Handle both dict and object access
                fid = f['id'] if isinstance(f, dict) else f.id
                ftitle = f['title'] if isinstance(f, dict) else f.title
                if fid == source_item.ownerFolder: return ftitle
    except: pass
    return None

def ensure_target_folder_exists(username, folder_name):
    try:
        user = target_gis.users.get(username)
        if not user: return False
        
        # Check existing folders (Handles Folder Objects AND Dicts)
        existing_folders = []
        for f in user.folders:
            if hasattr(f, 'title'):
                existing_folders.append(f.title)
            elif isinstance(f, dict):
                existing_folders.append(f.get('title'))
        
        if folder_name in existing_folders: return True
        
        # Create if missing
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except Exception as e: 
        print(f"      ⚠ Folder creation error: {e}")
        return False

def assign_correct_owner_and_folder(source_item, target_item):
    try:
        source_owner = source_item.owner
        target_owner_to_use = DEFAULT_OWNER
        target_folder_to_use = ACTIVE_FOLDER # Use verified folder

        found_user = target_gis.users.get(source_owner)
        
        if found_user:
            print(f"      👤 Source owner '{source_owner}' found in target.")
            target_owner_to_use = source_owner
            src_folder_name = get_source_folder_name(source_item)
            if src_folder_name:
                if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                    target_folder_to_use = src_folder_name
                else:
                    print(f"      ⚠️ Could not create folder '{src_folder_name}'. Using Default.")
            else:
                target_folder_to_use = None
        else:
            print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
            
        print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
        target_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)

    except Exception as e:
        print(f"      ⚠️ Reassign/Move Failed: {e} (Item remains with Creator)")

def copy_thumbnail(source_item, target_item):
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = "thumbnails_temp"
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path: target_item.update(thumbnail=thumb_path)
    except: pass

# =============================================================================
# --- HELPER: MIRROR SHARING (STRICT SOURCE MATCH + FIXES) ---------------------
# =============================================================================
def mirror_source_sharing(source_item, target_item):
    """
    1. Checks Source Sharing (Public/Org/Private).
    2. STRICTLY applies the same level to Target (No forcing Org).
    3. Maps Source Groups -> Target Groups (by exact Title).
    """
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")
        
        # 1. Access Level
        is_public = False
        is_org = False
        
        if source_item.access == 'public':
            is_public = True
            is_org = True 
        elif source_item.access == 'org':
            is_org = True
        # Else: Private (Both remain False)
            
        # 2. Map Groups (Robust retrieval)
        source_groups = []
        try:
            # Try getting via 'sharing.groups'
            raw_groups = source_item.sharing.groups
            # FIX: Explicitly check if it's a list. If not (e.g. Manager object), fail to except block.
            if isinstance(raw_groups, list):
                source_groups = raw_groups
            else:
                raise ValueError("Not a list")
        except:
            # Fallback to deprecated but reliable 'shared_with'
            # 'shared_with' usually returns a dict {'groups': [...]} or a list
            raw_shared = source_item.shared_with
            if isinstance(raw_shared, dict) and 'groups' in raw_shared:
                source_groups = raw_shared['groups']
            elif isinstance(raw_shared, list):
                source_groups = raw_shared

        target_group_ids = []
        
        if source_groups:
            print(f"         - Found {len(source_groups)} source groups. Mapping...")
            for sg in source_groups:
                # Handle Group Object vs String Name
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)

                # Search for group with exact title in Target
                # FIX: Removed 'max_items', using standard query
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                
                # Filter for exact match
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")
        
        # 3. Apply
        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- CORE LOGIC: ID TRACKER & GAP FIX -----------------------------------------
# =============================================================================
CACHE_TARGET_URLS = {}
CACHE_SOURCE_ID_LOOKUP = {}

def get_target_service_url(target_id):
    if target_id in CACHE_TARGET_URLS: return CACHE_TARGET_URLS[target_id]
    try:
        item = target_gis.content.get(target_id)
        if item:
            url = item.url
            CACHE_TARGET_URLS[target_id] = url
            return url
    except: pass
    return None

def find_source_id_from_url(url):
    base_url_raw = url.split("/FeatureServer")[0] + "/FeatureServer"
    if "/MapServer" in url: base_url_raw = url.split("/MapServer")[0] + "/MapServer"
    if base_url_raw in CACHE_SOURCE_ID_LOOKUP: return CACHE_SOURCE_ID_LOOKUP[base_url_raw]
    try:
        items = source_gis.content.search(f'url:"{base_url_raw}"', max_items=1)
        if items: 
            found_id = items[0].id
            CACHE_SOURCE_ID_LOOKUP[base_url_raw] = found_id
            return found_id
    except: pass
    return None

def process_layer_list(layer_list):
    for layer in layer_list:
        if "layers" in layer: process_layer_list(layer["layers"])
        
        url = layer.get("url", "")
        src_id = find_source_id_from_url(url)
        
        if src_id:
            if src_id in ID_MAP:
                target_id = ID_MAP[src_id]
                target_url_base = get_target_service_url(target_id)
                
                if target_url_base:
                    print(f"     ✅ Ledger Match: {layer.get('title')}")
                    print(f"        -> Source ID: {src_id} | Target ID: {target_id}")

                    idx = 0
                    try: idx = int(url.split("/")[-1])
                    except: pass
                    
                    new_idx = idx
                    if src_id == PROBLEM_SOURCE_ID:
                        if idx >= 18:
                            new_idx = idx - 1
                            print(f"        🔧 Gap Fix Applied: Shifting Index {idx} -> {new_idx}")
                        else:
                            print(f"        🔹 No Shift (Index {idx} <= 16)")
                    
                    layer["itemId"] = target_id
                    layer["url"] = f"{target_url_base}/{new_idx}"
                    layer["layerType"] = "ArcGISFeatureLayer"
                    if "serviceToken" in layer: del layer["serviceToken"]
                    STATS["layers_rewired"] += 1
                else:
                     print(f"     ⚠️ Ledger has ID {target_id}, but item not found in Target!")
            else:
                print(f"     ⚠️ No mapping found in CSV for Source ID: {src_id}")
        else:
             pass 

        if "layerDefinition" not in layer: layer["layerDefinition"] = {}
        if layer.get("layerType") == "ArcGISFeatureLayer":
             layer["layerDefinition"]["outFields"] = ["*"] 

def sanitize_basemap(map_json):
    is_deprecated = False
    bm_title = map_json.get("baseMap", {}).get("title", "").lower()
    if "deprecated" in bm_title or "world topographic" in bm_title:
        is_deprecated = True
    for lyr in map_json.get("baseMap", {}).get("baseMapLayers", []):
        if "deprecated" in lyr.get("url", "").lower():
            is_deprecated = True
            break
    if is_deprecated:
        print("   ⚠️ DETECTED DEPRECATED BASEMAP. Swapping to standard 'Topographic'...")
        map_json["baseMap"] = {
            "baseMapLayers": [{
                "id": "World_Topo_Map",
                "layerType": "ArcGISTiledMapServiceLayer",
                "url": "https://services.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer",
                "visibility": True,
                "opacity": 1,
                "title": "Topographic"
            }],
            "title": "Topographic"
        }
    return map_json

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"Starting V34.3 Migration (Strict Mirroring + Groups)...")
start_time = time.time()

for map_id in WEB_MAP_IDS_TO_MIGRATE:
    try:
        STATS["maps_scanned"] += 1
        
        if str(map_id) in ALREADY_MIGRATED_IDS:
             print(f"\n[Skip] Map {map_id} found in History Log.")
             STATS["maps_skipped_log"] += 1
             continue

        source_web_map = source_gis.content.get(map_id)
        if not source_web_map:
            print(f"❌ Map {map_id} not found.")
            STATS["maps_skipped_missing_data"] += 1
            STATS["failures"] += 1
            continue

        print(f"\nProcessing Map: {source_web_map.title}...")

        target_title = str(source_web_map.title) if source_web_map.title else "Untitled Map"
        if APPEND_MIGRATED:
            target_title = f"{target_title} (Migrated)"

        existing = target_gis.content.search(f'title:"{target_title}"', max_items=1)
        if existing:
            print(f"   ⚠️ Map already exists in Target. Skipping.")
            STATS["maps_skipped_log"] += 1
            continue
        
        map_json = source_web_map.get_data()
        new_json = copy.deepcopy(map_json)
        if "operationalLayers" in new_json: process_layer_list(new_json["operationalLayers"])
        new_json = sanitize_basemap(new_json)

        print("   1. Creating Web Map item...")
        
        snippet_val = str(source_web_map.snippet) if source_web_map.snippet else ""
        tags_list = ["Migrated", f"SourceID:{map_id}"]
        tags_str = ",".join(tags_list) 
        
        item_props = {
            "type": "Web Map",
            "title": target_title,
            "snippet": snippet_val,
            "tags": tags_str,
            "text": json.dumps(new_json)
        }
        
        # USE THE VERIFIED FOLDER (Or None if verification failed)
        new_map = target_gis.content.add(item_props, folder=ACTIVE_FOLDER)

        if new_map:
            copy_full_metadata(source_web_map, new_map, map_id)
            copy_thumbnail(source_web_map, new_map)
            
            # --- STRICT SHARING MIRROR ---
            # CALLED BEFORE REASSIGNING OWNER TO AVOID PRIVILEGE ERRORS
            mirror_source_sharing(source_web_map, new_map)
            
            assign_correct_owner_and_folder(source_web_map, new_map)

            log_migration(map_id, "N/A", new_map.id, new_map.title, "Web Map")
            STATS["maps_created"] += 1
            print(f"🚀 SUCCESS: Created Map '{new_map.title}'")
        
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ Failed processing map {map_id}: {e}")
        STATS["failures"] += 1

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "="*60)
print("        WEB MAP MIGRATION REPORT (V34.3)        ")
print("="*60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 🗺️  Maps Scanned:               {STATS['maps_scanned']}")
print(f" 🧾 Skipped (log):             {STATS['maps_skipped_log']}")
print(f" 🛑 Skipped (missing data):    {STATS['maps_skipped_missing_data']}")
print(f" ✅ Maps Created:              {STATS['maps_created']}")
print(f" 🔗 Layers Rewired:            {STATS['layers_rewired']}")
print(f" ❌ Failures:                  {STATS['failures']}")
print("="*60)